In [1]:
!pip install dlib opencv-python numpy pandas matplotlib

In [2]:
import cv2
import dlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from google.colab import files

In [3]:
!wget -q http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bunzip2 -f shape_predictor_68_face_landmarks.dat.bz2

In [4]:
print(os.path.exists("shape_predictor_68_face_landmarks.dat"))

True


In [5]:
face_detector = dlib.get_frontal_face_detector()

landmark_predictor = dlib.shape_predictor(
    "shape_predictor_68_face_landmarks.dat"
)

print("Dlib models loaded successfully!")

Dlib models loaded successfully!


In [ ]:
uploaded = files.upload()

In [ ]:
person_classes = {
    "Rahaf_1.jpg": 0,
    "Rahaf_2.jpg": 0,
    "Rahaf_3.jpg": 0,
    "Rahaf_4.jpg": 0,
    "Rahaf_5.jpg": 0,
    "Rahaf_6.jpg": 0,
    "Rahaf_7.jpg": 0,
    "Rahaf_8.jpg": 0,
    "Rahaf_9.jpg": 0,
    "Rahaf_10.jpg": 0,
    "Rahaf_11.jpg": 0,
    "Rahaf_12.jpg": 0,
    "Rahaf_13.jpg": 0,
    "Rahaf_14.jpg": 0,
    "Rahaf_15.jpg": 0,
    "Rahaf_16.jpg": 0,
    "Rahaf_17.jpg": 0,
    "Rahaf_18.jpg": 0,
    "Rahaf_19.jpg": 0,
    "Rahaf_20.jpg": 0,

    "Haneen_1.jpg": 1,
    "Haneen_2.jpg": 1,
    "Haneen_3.jpg": 1,
    "Haneen_4.jpg": 1,
    "Haneen_5.jpg": 1,
    "Haneen_6.jpg": 1,
    "Haneen_7.jpg": 1,
    "Haneen_8.jpg": 1,
    "Haneen_9.jpg": 1,
    "Haneen_10.jpg": 1,
    "Haneen_11.jpg": 1,
    "Haneen_12.jpg": 1,
    "Haneen_13.jpg": 1,
    "Haneen_14.jpg": 1,
    "Haneen_15.jpg": 1,
    "Haneen_16.jpg": 1,
    "Haneen_17.jpg": 1,
    "Haneen_18.jpg": 1,
    "Haneen_19.jpg": 1,
    "Haneen_20.jpg": 1


}

In [ ]:
def extract_landmarks(image_path):

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        print("Could not read:", image_path)
        return None, None

    # Convert BGR to RGB
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Detect faces
    faces = face_detector(rgb_image, 2)

    if len(faces) == 0:
        print("No face detected in:", image_path)
        return None, None

    # Use the first detected face
    face = faces[0]

    # Detect 68 facial landmarks
    landmarks = landmark_predictor(rgb_image, face)

    # Store coordinates
    features = []

    for i in range(68):
        x = landmarks.part(i).x
        y = landmarks.part(i).y

        features.append(x)
        features.append(y)

    return features, landmarks

In [ ]:
def draw_landmarks(image_path, landmarks):

    image = cv2.imread(image_path)

    # Convert BGR to RGB for displaying
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Draw all 68 landmarks
    for i in range(68):

        x = landmarks.part(i).x
        y = landmarks.part(i).y

        cv2.circle(
            image_rgb,
            (x, y),
            2,
            (0, 255, 0),
            -1
        )

    return image_rgb

In [ ]:
dataset = []

for image_name, class_label in person_classes.items():

    features, landmarks = extract_landmarks(image_name)

    if features is None:
        continue

    # Add class label
    features_with_class = features + [class_label]

    dataset.append(features_with_class)

    print(
        image_name,
        "->",
        len(features),
        "features",
        "Class:",
        class_label
    )

print("\nTotal faces processed:", len(dataset))

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# Create a folder to save the landmark images
output_folder = "landmark_images"
os.makedirs(output_folder, exist_ok=True)

for image_name, class_label in person_classes.items():

    # Extract landmarks
    features, landmarks = extract_landmarks(image_name)

    if landmarks is None:
        continue

    # Draw the 68 landmarks
    image_with_landmarks = draw_landmarks(
        image_name,
        landmarks
    )

    # Create output filename
    base_name = os.path.splitext(image_name)[0]
    output_name = base_name + "_landmarks.jpg"
    output_path = os.path.join(output_folder, output_name)

    # Convert RGB to BGR before saving
    image_bgr = cv2.cvtColor(
        image_with_landmarks,
        cv2.COLOR_RGB2BGR
    )

    # Save the image
    cv2.imwrite(output_path, image_bgr)

    # Display the image
    plt.figure(figsize=(6, 6))
    plt.imshow(image_with_landmarks)
    plt.title(
        f"{image_name} - Class {class_label} - 68 Landmarks"
    )
    plt.axis("off")
    plt.show()

    print("Saved:", output_path)

In [ ]:
from google.colab import files
import shutil

# Compress the folder into a ZIP file
shutil.make_archive(
    "landmark_images",
    "zip",
    "landmark_images"
)

# Download the ZIP file
files.download("landmark_images.zip")

In [ ]:
columns = []

for i in range(1, 69):
    columns.append(f"x{i}")
    columns.append(f"y{i}")

columns.append("Class")

print("Number of columns:", len(columns))

In [ ]:
df = pd.DataFrame(
    dataset,
    columns=columns
)

df.head(20)

In [ ]:
print("Number of facial landmark features:", len(columns) - 1)
print("Expected:", 136)

print("\nDataset shape:")
print(df.shape)

In [ ]:
csv_filename = "facial_landmarks_dataset.csv"

df.to_csv(
    csv_filename,
    index=False
)

print("Dataset saved successfully!")
print(csv_filename)

In [ ]:
from google.colab import files

files.download("facial_landmarks_dataset.csv")

**TASK 2**

In [ ]:
import pandas as pd
import numpy as np
import cv2
import dlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv("facial_landmarks_dataset.csv")

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["Class"].value_counts())

df.head()

In [ ]:
import pandas as pd

df = pd.read_csv("facial_landmarks_dataset.csv")

X = df.iloc[:, :-1]
y = df["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
def normalize_landmark_features(X):

    normalized_data = []

    for row in X.values:

        x_coords = row[0::2]
        y_coords = row[1::2]

        min_x = x_coords.min()
        max_x = x_coords.max()

        min_y = y_coords.min()
        max_y = y_coords.max()

        width = max_x - min_x
        height = max_y - min_y

        if width == 0 or height == 0:
            continue

        normalized = []

        for i in range(68):

            nx = (x_coords[i] - min_x) / width
            ny = (y_coords[i] - min_y) / height

            normalized.append(nx)
            normalized.append(ny)

        normalized_data.append(normalized)

    return np.array(normalized_data)

In [ ]:
X_normalized = normalize_landmark_features(X)

print("Original shape:", X.shape)
print("Normalized shape:", X_normalized.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

In [ ]:
perceptron = Perceptron(
    max_iter=1000,
    random_state=42
)

perceptron.fit(
    X_train_scaled,
    y_train
)

print("Perceptron training completed.")

In [ ]:
y_pred = perceptron.predict(X_test_scaled)

print("Actual classes:   ", y_test.to_numpy())
print("Predicted classes:", y_pred)

In [ ]:
from sklearn.metrics import accuracy_score

y_train_pred = perceptron.predict(
    X_train_scaled
)

y_test_pred = perceptron.predict(
    X_test_scaled
)

train_accuracy = accuracy_score(
    y_train,
    y_train_pred
)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(
    "Training Accuracy:",
    f"{train_accuracy * 100:.2f}%"
)

print(
    "Testing Accuracy:",
    f"{test_accuracy * 100:.2f}%"
)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

In [ ]:
plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Rahaf", "Haneen"],
    yticklabels=["Rahaf", "Haneen"]
)

plt.xlabel("Predicted Person")
plt.ylabel("Actual Person")
plt.title("Perceptron Confusion Matrix")

plt.show()

In [ ]:
print("Classification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Rahaf", "Haneen"],
        zero_division=0
    )
)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
face_detector = dlib.get_frontal_face_detector()

landmark_predictor = dlib.shape_predictor(
    "shape_predictor_68_face_landmarks.dat"
)

print("Dlib models loaded.")

In [ ]:
class_names = {
    0: "Rahaf",
    1: "Haneen"
}

In [ ]:
def get_face_features(image):
    if image is None:
        print("Input image is None.")
        return None, None, None # Return features, landmarks, face

    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    faces = face_detector(rgb_image, 2)

    if len(faces) == 0:
        print("No face detected.")
        return None, None, None # Return features, landmarks, face

    face = faces[0]
    landmarks = landmark_predictor(rgb_image, face)

    features = []
    for i in range(68):
        x = landmarks.part(i).x
        y = landmarks.part(i).y
        features.append(x)
        features.append(y)

    return features, landmarks, face
# List of test images
test_images = [
    "Test.jpg",
    "Test_1.jpg",
    "Test_3.jpg",
    "Test_4.jpg"
]

# Test each image
for image_path in test_images:

    print("\n==============================")
    print("Testing:", image_path)
    print("==============================")

    # Read image
    image = cv2.imread(image_path)

    # Extract face features
    features, landmarks, face = get_face_features(image)

    if features is None:
        print("Result: No face detected.")

    else:
        # Convert features to DataFrame
        features_df = pd.DataFrame(
            [features],
            columns=X.columns
        )

        # Apply the SAME normalization used during training
        features_normalized = normalize_landmark_features(features_df)

        # Apply the SAME scaler used during training
        features_scaled = scaler.transform(features_normalized)

        # Predict person
        predicted_class = perceptron.predict(features_scaled)[0]

        predicted_name = class_names[predicted_class]

        print("Predicted Person:", predicted_name)
        print("Predicted Class:", predicted_class)



In [ ]:
def euclidean_distance(p1, p2):

    return np.linalg.norm(
        np.array(p1) - np.array(p2)
    )

In [ ]:
def calculate_ear(landmarks):

    # Left eye landmarks: 36-41
    left_eye = [
        (landmarks.part(i).x, landmarks.part(i).y)
        for i in range(36, 42)
    ]

    # Right eye landmarks: 42-47
    right_eye = [
        (landmarks.part(i).x, landmarks.part(i).y)
        for i in range(42, 48)
    ]

    # Left eye EAR
    left_ear = (
        euclidean_distance(left_eye[1], left_eye[5]) +
        euclidean_distance(left_eye[2], left_eye[4])
    ) / (
        2 * euclidean_distance(left_eye[0], left_eye[3])
    )

    # Right eye EAR
    right_ear = (
        euclidean_distance(right_eye[1], right_eye[5]) +
        euclidean_distance(right_eye[2], right_eye[4])
    ) / (
        2 * euclidean_distance(right_eye[0], right_eye[3])
    )

    # Average EAR
    ear = (left_ear + right_ear) / 2.0

    return ear

In [ ]:
EAR_THRESHOLD = 0.21

In [ ]:
ear = calculate_ear(landmarks)

if ear > EAR_THRESHOLD:
    eye_status = "Open"
else:
    eye_status = "Closed"

print(f"EAR: {ear:.3f}")
print("Eye Status:", eye_status)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
video_paths = [
    "Testing_video.MP4"

]

print("Videos loaded:")
for video in video_paths:
    print("-", video)


In [ ]:
import cv2

video_paths = [
    "Testing_video.MP4",

]

for video_path in video_paths:

    print("\n" + "=" * 50)
    print("Testing:", video_path)

    cap = cv2.VideoCapture(video_path)

    print("Is opened:", cap.isOpened())
    print("FPS:", cap.get(cv2.CAP_PROP_FPS))
    print("Number of frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))

    cap.release()


In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output
from PIL import Image
from collections import Counter

EAR_THRESHOLD = 0.21
SLEEP_SECONDS = 2

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

CLOSED_FRAMES_REQUIRED = int(fps * SLEEP_SECONDS)

closed_frames = 0
person_predictions = []

last_ear = 0
last_eye_status = "Unknown"
last_person_status = "Awake"

print("Video FPS:", fps)
print("Frames required for sleeping:", CLOSED_FRAMES_REQUIRED)

if not cap.isOpened():
    print("Error: Could not open video.")

else:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        faces = face_detector(rgb_frame, 1)

        if len(faces) > 0:

            face = faces[0]

            landmarks = landmark_predictor(
                rgb_frame,
                face
            )

            # --------------------------------
            # Extract 136 normalized features
            # --------------------------------

            x_coords = []
            y_coords = []

            for i in range(68):
                x_coords.append(landmarks.part(i).x)
                y_coords.append(landmarks.part(i).y)

            min_x = min(x_coords)
            max_x = max(x_coords)

            min_y = min(y_coords)
            max_y = max(y_coords)

            width = max_x - min_x
            height = max_y - min_y

            if width == 0 or height == 0:
                continue

            features = []

            for i in range(68):

                nx = (x_coords[i] - min_x) / width
                ny = (y_coords[i] - min_y) / height

                features.append(nx)
                features.append(ny)

            features = np.array(features).reshape(1, -1)

            # --------------------------------
            # Scale features
            # --------------------------------

            features_scaled = scaler.transform(features)

            # --------------------------------
            # Predict person
            # --------------------------------

            predicted_class = perceptron.predict(
                features_scaled
            )[0]

            predicted_name = class_names.get(
                predicted_class,
                "Unknown"
            )

            person_predictions.append(
                predicted_name
            )

            # --------------------------------
            # Calculate EAR
            # --------------------------------

            ear = calculate_ear(landmarks)

            last_ear = ear

            # --------------------------------
            # Eye status
            # --------------------------------

            if ear > EAR_THRESHOLD:

                eye_status = "Open"

                closed_frames = 0

            else:

                eye_status = "Closed"

                closed_frames += 1

            last_eye_status = eye_status

            # --------------------------------
            # Sleep detection
            # --------------------------------

            if closed_frames >= CLOSED_FRAMES_REQUIRED:

                person_status = "Sleeping"

            else:

                person_status = "Awake"

            last_person_status = person_status

            # --------------------------------
            # Draw landmarks
            # --------------------------------

            for i in range(68):

                x = landmarks.part(i).x
                y = landmarks.part(i).y

                cv2.circle(
                    frame,
                    (x, y),
                    2,
                    (0, 255, 0),
                    -1
                )

            # --------------------------------
            # Display information
            # --------------------------------

            cv2.putText(
                frame,
                f"Person: {predicted_name}",
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"EAR: {ear:.3f}",
                (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Eye Status: {eye_status}",
                (20, 90),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Status: {person_status}",
                (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            clear_output(wait=True)

            display(
                Image.fromarray(
                    cv2.cvtColor(
                        frame,
                        cv2.COLOR_BGR2RGB
                    )
                )
            )

    cap.release()

    # --------------------------------
    # Final person prediction
    # --------------------------------

    if len(person_predictions) > 0:

        person_counts = Counter(
            person_predictions
        )

        final_person = person_counts.most_common(1)[0][0]

    else:

        final_person = "Unknown"

    print("\n")
    print("=" * 50)
    print("              FINAL RESULT")
    print("=" * 50)

    print("Person:", final_person)
    print("Final EAR:", round(last_ear, 3))
    print("Eye Status:", last_eye_status)
    print("Final Status:", last_person_status)

    print("=" * 50)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
video_paths = [
    "IMG_6299 (1).MOV"

]

print("Videos loaded:")
for video in video_paths:
    print("-", video)


In [ ]:
import cv2

video_paths = [
    "IMG_6299 (1).MOV",

]

for video_path in video_paths:

    print("\n" + "=" * 50)
    print("Testing:", video_path)

    cap = cv2.VideoCapture(video_path)

    print("Is opened:", cap.isOpened())
    print("FPS:", cap.get(cv2.CAP_PROP_FPS))
    print("Number of frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))

    cap.release()


In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output
from PIL import Image
from collections import Counter

EAR_THRESHOLD = 0.21
SLEEP_SECONDS = 2

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

CLOSED_FRAMES_REQUIRED = int(fps * SLEEP_SECONDS)

closed_frames = 0
person_predictions = []

last_ear = 0
last_eye_status = "Unknown"
last_person_status = "Awake"

print("Video FPS:", fps)
print("Frames required for sleeping:", CLOSED_FRAMES_REQUIRED)

if not cap.isOpened():
    print("Error: Could not open video.")

else:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        faces = face_detector(rgb_frame, 1)

        if len(faces) > 0:

            face = faces[0]

            landmarks = landmark_predictor(
                rgb_frame,
                face
            )

            # --------------------------------
            # Extract 136 normalized features
            # --------------------------------

            x_coords = []
            y_coords = []

            for i in range(68):
                x_coords.append(landmarks.part(i).x)
                y_coords.append(landmarks.part(i).y)

            min_x = min(x_coords)
            max_x = max(x_coords)

            min_y = min(y_coords)
            max_y = max(y_coords)

            width = max_x - min_x
            height = max_y - min_y

            if width == 0 or height == 0:
                continue

            features = []

            for i in range(68):

                nx = (x_coords[i] - min_x) / width
                ny = (y_coords[i] - min_y) / height

                features.append(nx)
                features.append(ny)

            features = np.array(features).reshape(1, -1)

            # --------------------------------
            # Scale features
            # --------------------------------

            features_scaled = scaler.transform(features)

            # --------------------------------
            # Predict person
            # --------------------------------

            predicted_class = perceptron.predict(
                features_scaled
            )[0]

            predicted_name = class_names.get(
                predicted_class,
                "Unknown"
            )

            person_predictions.append(
                predicted_name
            )

            # --------------------------------
            # Calculate EAR
            # --------------------------------

            ear = calculate_ear(landmarks)

            last_ear = ear

            # --------------------------------
            # Eye status
            # --------------------------------

            if ear > EAR_THRESHOLD:

                eye_status = "Open"

                closed_frames = 0

            else:

                eye_status = "Closed"

                closed_frames += 1

            last_eye_status = eye_status

            # --------------------------------
            # Sleep detection
            # --------------------------------

            if closed_frames >= CLOSED_FRAMES_REQUIRED:

                person_status = "Sleeping"

            else:

                person_status = "Awake"

            last_person_status = person_status

            # --------------------------------
            # Draw landmarks
            # --------------------------------

            for i in range(68):

                x = landmarks.part(i).x
                y = landmarks.part(i).y

                cv2.circle(
                    frame,
                    (x, y),
                    2,
                    (0, 255, 0),
                    -1
                )

            # --------------------------------
            # Display information
            # --------------------------------

            cv2.putText(
                frame,
                f"Person: {predicted_name}",
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"EAR: {ear:.3f}",
                (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Eye Status: {eye_status}",
                (20, 90),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Status: {person_status}",
                (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            clear_output(wait=True)

            display(
                Image.fromarray(
                    cv2.cvtColor(
                        frame,
                        cv2.COLOR_BGR2RGB
                    )
                )
            )

    cap.release()

    # --------------------------------
    # Final person prediction
    # --------------------------------

    if len(person_predictions) > 0:

        person_counts = Counter(
            person_predictions
        )

        final_person = person_counts.most_common(1)[0][0]

    else:

        final_person = "Unknown"

    print("\n")
    print("=" * 50)
    print("              FINAL RESULT")
    print("=" * 50)

    print("Person:", final_person)
    print("Final EAR:", round(last_ear, 3))
    print("Eye Status:", last_eye_status)
    print("Final Status:", last_person_status)

    print("=" * 50)